# ZPDES progression by module-specific initial Elo quartile

This notebook asks whether students show similar progression in ZPDES regardless of their **initial level within the module**.

A student's initial adaptive-test Elo and quartile are specific to the studentâ€“module pair. The same student can therefore be Q4 in one module and Q1 in another.

The analysis uses one row per studentâ€“module pair: `mean_progress` is averaged across that pair's eligible ZPDES activity sequences. This prevents a student with many eligible sequences in one module from receiving more rows for that module.

The requested model is fitted as:

```text
mean_progress ~ elo_quartile
  + classroom random intercept
  + student random intercept
  + module random intercept
  + independent module random deviations for Q2, Q3, and Q4
```

This is the scalable independent-random-coefficient analogue of `(1 + elo_quartile | module_title)`. It returns one average Q2â€“Q1, Q3â€“Q1, and Q4â€“Q1 comparison while allowing those comparisons to vary between modules.

## 1. Setup

In [1]:
from __future__ import annotations

import importlib
import sys
from argparse import Namespace
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.model_work_mode_progress as work_mode_model  # noqa: E402
import scripts.model_zpdes_elo_quartile_equity as quartile_model  # noqa: E402

importlib.reload(work_mode_model)
importlib.reload(quartile_model)

from scripts.model_work_mode_progress import build_activity_level, load_attempts  # noqa: E402
from scripts.model_zpdes_elo_quartile_equity import (  # noqa: E402
    build_student_module_analysis,
    fit_quartile_equity_model,
    load_adaptive_test_elo,
    load_module_lookup,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

## 2. Parameters

`MIN_ADAPTIVE_TEST_ATTEMPTS = 1` matches the no-minimum analysis discussed previously. Increase it and rerun the notebook as a sensitivity analysis if Elo estimates based on very few adaptive-test answers are a concern.

`EQUIVALENCE_MARGIN_POINTS` is a substantive threshold, not a value learned from the data. Replace `3.0` if another difference is considered educationally negligible.

In [2]:
INPUT_FILE = PROJECT_ROOT / "data_MIA" / "986-neurips-mia_20260415_100024.parquet"
EXERCISE_CATALOG_JSON = PROJECT_ROOT / "data_MIA" / "exo_mia.json"
MODULE_CONFIG_JSON = PROJECT_ROOT / "data_MIA" / "config_mia.json"
ADAPTIVE_ELO_FILE = (
    PROJECT_ROOT / "artifacts" / "reports" / "mia_adaptive_test_elo_before_practice.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "zpdes_elo_quartile_equity_notebook"

MIN_ACTIVITY_EXERCISES = 4
MIN_ADAPTIVE_TEST_ATTEMPTS = 1
MIN_STUDENT_MODULE_PAIRS_PER_QUARTILE = 20
EQUIVALENCE_MARGIN_POINTS = 3.0
MAXITER = 300
TRACE_OPTIMIZER = False
RUN_MODEL = True

for required_path in (
    INPUT_FILE,
    EXERCISE_CATALOG_JSON,
    MODULE_CONFIG_JSON,
    ADAPTIVE_ELO_FILE,
):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input not found: {required_path}")

args = Namespace(
    input_file=INPUT_FILE,
    input_csv=None,
    data_dir=None,
    exercise_catalog_json=EXERCISE_CATALOG_JSON,
    module_config_json=MODULE_CONFIG_JSON,
    keep_only_single_module_playlists=False,
)

## 3. Rebuild eligible sequence progression

The progression construction matches the existing work-mode analysis: only the first retained attempt at each exercise is used, and an activity sequence must contain at least four unique exercises. The outcome is the later-half success rate minus the first-half success rate, in percentage points.

In [3]:
attempts = load_attempts(args)
activity_level = build_activity_level(
    attempts,
    min_activity_exercises=MIN_ACTIVITY_EXERCISES,
    exercise_elo=None,
)

attempt_summary = pd.DataFrame(
    [
        {
            "attempt_rows": len(attempts),
            "students": attempts["student_id"].nunique(),
            "classrooms": attempts["classroom_id"].nunique(),
            "modules": attempts["module"].nunique(),
            "eligible_sequence_rows": len(activity_level),
            "eligible_zpdes_sequence_rows": int(activity_level["work_mode"].eq("zpdes").sum()),
        }
    ]
)
display(attempt_summary)

,attempt_rows,students,classrooms,modules,eligible_sequence_rows,eligible_zpdes_sequence_rows
0,5590740,37894,3091,27,468013,349909


## 4. Attach module-specific initial Elo and assign quartiles

First, eligible ZPDES sequence progression is averaged within each studentâ€“module pair. The adaptive-test Elo is then joined on both `student_id` and `module_code`. Quartiles are calculated separately within each module among the eligible studentâ€“module pairs.

Equal Elo values receive the same percentile rank, so tied students are not arbitrarily split across quartiles.

In [4]:
module_lookup = load_module_lookup(MODULE_CONFIG_JSON)
adaptive_elo = load_adaptive_test_elo(ADAPTIVE_ELO_FILE, source_id="mia")

analysis_data, sample_audit, module_coverage = build_student_module_analysis(
    activity_level,
    adaptive_elo,
    module_lookup,
    min_adaptive_test_attempts=MIN_ADAPTIVE_TEST_ATTEMPTS,
    min_pairs_per_quartile=MIN_STUDENT_MODULE_PAIRS_PER_QUARTILE,
)

display(sample_audit)
display(
    module_coverage.sort_values(
        ["eligible", "minimum_quartile_pairs"], ascending=[False, False]
    )
)

,zpdes_sequence_rows,zpdes_students,zpdes_modules,student_module_pairs_before_elo,pairs_with_eligible_elo,pairs_retained_for_model,students_retained,classrooms_retained,modules_retained,min_adaptive_test_attempts,min_pairs_per_quartile
0,349909,25489,27,39693,35432,35431,23835,2419,21,1,20


elo_quartile,module_code,module_title,Q1,Q2,Q3,Q4,minimum_quartile_pairs,eligible
2,M101,Réapprentissage du sens des nombres,2406,2407,2406,2407,2406,True
18,M6,Syntaxe niveau 1,838,838,838,839,838,True
3,M102,Comprendre les notions de proportion et de fraction,804,804,804,805,804,True
20,M8,Orthographe niveau 1,801,802,802,802,801,True
17,M4,Améliorer la compréhension des textes,591,592,592,592,591,True
6,M105,"Organisation et gestion de données, fonctions",471,472,471,472,471,True
0,M1,Réapprentissage des correspondances graphèmes-phonèmes,414,415,415,415,414,True
10,M12,Verbe niveau 1,369,370,369,370,369,True
8,M107,Espace et Géométrie,352,352,352,352,352,True
4,M103,Nombres et calculs,339,340,340,340,339,True


## 5. Verify that quartile is genuinely module-specific

The table below counts how many retained students appear in one versus several different quartiles across modules. A student observed in Q1 in one module and Q4 in another is handled as two studentâ€“module observations sharing the same student random intercept.

In [5]:
student_quartile_mobility = (
    analysis_data.groupby("student_id", as_index=False)
    .agg(
        modules_observed=("module_code", "nunique"),
        distinct_quartiles=("elo_quartile", "nunique"),
    )
)

mobility_summary = (
    student_quartile_mobility.groupby("distinct_quartiles", as_index=False)
    .agg(students=("student_id", "size"))
    .sort_values("distinct_quartiles")
)
display(mobility_summary)
display(analysis_data.head())

,distinct_quartiles,students
0,1,18595
1,2,4357
2,3,800
3,4,83


,student_id,classroom_id,module_code,module_title,mean_progress,n_sequences,n_activities,adaptive_test_attempts,adaptive_test_elo,elo_quartile
0,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,M4,Améliorer la compréhension des textes,22.916667,12,12,14.0,1633.704396,Q4
1,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,M8,Orthographe niveau 1,16.666667,9,9,15.0,1536.885126,Q2
2,00017460-1206-45ee-b1d3-393c72a45220,132484cb-02b9-4cc2-974a-e38e485dd1f1,M9,Orthographe niveau 2,10.416667,8,8,13.0,1557.482630,Q2
3,000aa84a-30dd-4dd9-bcfc-bfbb000be34f,544946a0-9490-4720-a9ac-319e26018328,M107,Espace et Géométrie,12.500000,2,2,13.0,1548.626433,Q3
4,000b4ef0-febe-4830-9c2d-a3945b3a8bd9,adcec886-5e67-4bb8-a0be-bb1411ff0829,M101,Réapprentissage du sens des nombres,50.000000,6,6,30.0,1748.558138,Q3


## 6. Descriptive progression by quartile

These are unadjusted studentâ€“module means. They are useful for orientation but are not the mixed-model results.

In [6]:
quartile_descriptive = (
    analysis_data.groupby("elo_quartile", as_index=False, observed=True)
    .agg(
        student_module_pairs=("mean_progress", "size"),
        students=("student_id", "nunique"),
        modules=("module_code", "nunique"),
        mean_progress=("mean_progress", "mean"),
        median_progress=("mean_progress", "median"),
        progress_sd=("mean_progress", "std"),
        mean_adaptive_test_elo=("adaptive_test_elo", "mean"),
        median_adaptive_test_attempts=("adaptive_test_attempts", "median"),
    )
)
display(quartile_descriptive.round(2))

,elo_quartile,student_module_pairs,students,modules,mean_progress,median_progress,progress_sd,mean_adaptive_test_elo,median_adaptive_test_attempts
0,Q1,8850,7301,21,15.47,16.00,20.14,1377.74,12.0
1,Q2,8861,7832,21,17.29,16.67,20.21,1531.80,13.0
2,Q3,8854,7724,21,18.85,18.75,20.71,1635.73,14.0
3,Q4,8866,7184,21,20.48,20.83,20.24,1781.11,15.0


## 7. Fit the crossed mixed model

The fixed effects estimate the average Q2â€“Q1, Q3â€“Q1, and Q4â€“Q1 differences across modules. The module random coefficients allow those differences to be positive in some modules and negative in others.

Random effects are set to their population-average value of zero when the adjusted quartile means are calculated.

In [7]:
fit = None
if RUN_MODEL:
    fit = fit_quartile_equity_model(
        analysis_data,
        equivalence_margin=EQUIVALENCE_MARGIN_POINTS,
        maxiter=MAXITER,
        trace=TRACE_OPTIMIZER,
    )
    display(fit.diagnostics)
else:
    print("RUN_MODEL is False; only data preparation and descriptive outputs were produced.")

C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,status,converged,finite_fixed_effect_standard_errors,iterations,maxiter,log_likelihood,n_student_module_pairs,n_students,n_classrooms,n_modules,model_specification
0,ok,True,True,11,300,-156505.14139,35431,23835,2419,21,"Gaussian mixed model: mean_progress ~ elo_quartile; random intercepts for classroom, student, and module; independent module random devi..."


## 8. Overall quartile results

The intercept is adjusted mean progression for Q1 in a typical module. `Q2`, `Q3`, and `Q4` are average differences from Q1 across modules.

The adjusted means are point estimates. The confidence intervals and p-values apply to each Q-versus-Q1 contrast. Holm-adjusted p-values account for the three planned quartile comparisons.

In [8]:
if fit is not None:
    display(fit.fixed_effects.round(4))
    display(fit.adjusted_quartile_means.round(3))

,term,estimate,std_error,z_value,p_value,ci_low,ci_high,p_value_holm
0,Intercept,15.0331,0.8003,18.7846,0.0000,13.4645,16.6016,NaN
1,Q2,1.8973,0.4930,3.8486,0.0001,0.9310,2.8635,0.0001
2,Q3,3.4888,0.5335,6.5398,0.0000,2.4432,4.5344,0.0000
3,Q4,4.3754,0.8330,5.2523,0.0000,2.7427,6.0082,0.0000


,elo_quartile,adjusted_mean_progress,difference_from_q1,difference_ci_low,difference_ci_high,difference_p_value,difference_p_value_holm,difference_95ci_inside_margin,equivalence_margin_points
0,Q1,15.033,0.000,NaN,NaN,NaN,NaN,<NA>,3.0
1,Q2,16.930,1.897,0.931,2.863,0.0,0.0,True,3.0
2,Q3,18.522,3.489,2.443,4.534,0.0,0.0,False,3.0
3,Q4,19.408,4.375,2.743,6.008,0.0,0.0,False,3.0


## 9. Does the quartile relationship vary between modules?

`module_random_slope_sd` summarizes how much each Q-versus-Q1 contrast varies between modules. The approximate module range is the average contrast plus or minus 1.96 random-slope standard deviations. It is a heterogeneity summary, not a confidence interval or a formal module-by-module test.

In [9]:
if fit is not None:
    display(fit.module_heterogeneity.round(3))
    display(fit.variance_components.round(5))

,contrast,average_difference,module_random_slope_sd,approx_module_difference_low,approx_module_difference_high
0,Q2 - Q1,1.897,1.485,-1.014,4.808
1,Q3 - Q1,3.489,1.709,0.138,6.839
2,Q4 - Q1,4.375,3.309,-2.110,10.861


,component,variance,std_deviation
0,Error_var,394.51967,19.86252
1,classroom_id,6.89071,2.62502
2,module_title,11.62953,3.41021
3,student_id,0.39919,0.63181
4,module_title_rand_coef_Q2,2.20556,1.48511
5,module_title_rand_coef_Q3,2.92234,1.70949
6,module_title_rand_coef_Q4,10.94798,3.30877


## 10. Interpretation for the equity question

Evidence consistent with equitable progression requires both:

1. positive adjusted progression in Q1, Q2, Q3, and Q4; and
2. Q2â€“Q1, Q3â€“Q1, and Q4â€“Q1 confidence intervals entirely inside the prespecified practical-equivalence margin.

The module random-slope standard deviations must also be considered: small average differences can hide positive associations in some modules and negative associations in others.

The 95% confidence-interval margin check below is a conservative descriptive criterion, not a formal TOST equivalence test.

This is an observational analysis of progression recorded under ZPDES. Without a playlist comparison or randomized assignment, phrase the conclusion as an association under observed ZPDES useâ€”not as proof that ZPDES caused equitable progression.

In [10]:
if fit is not None:
    equity_table = fit.adjusted_quartile_means.copy()
    all_point_estimates_positive = equity_table["adjusted_mean_progress"].gt(0).all()
    contrasts = equity_table.loc[equity_table["elo_quartile"].ne("Q1")]
    outside_margin = contrasts.loc[
        ~contrasts["difference_95ci_inside_margin"].astype(bool), "elo_quartile"
    ].tolist()

    print(f"All four adjusted progression point estimates are positive: {all_point_estimates_positive}.")
    if outside_margin:
        print(
            f"With a +/-{EQUIVALENCE_MARGIN_POINTS:.1f}-point practical margin, "
            "the 95% difference interval is not fully inside the margin for: "
            + ", ".join(outside_margin)
            + "."
        )
        print("The selected margin therefore does not support overall practical equivalence across all quartiles.")
    else:
        print("All three Q-versus-Q1 intervals lie inside the selected practical margin.")
    print("Interpret this together with the module random-slope variation shown above.")

All four adjusted progression point estimates are positive: True.
With a +/-3.0-point practical margin, the 95% difference interval is not fully inside the margin for: Q3, Q4.
The selected margin therefore does not support overall practical equivalence across all quartiles.
Interpret this together with the module random-slope variation shown above.


## 11. Save summary outputs

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample_audit.to_csv(OUTPUT_DIR / "sample_audit.csv", index=False, encoding="utf-8-sig")
module_coverage.to_csv(
    OUTPUT_DIR / "module_quartile_coverage.csv", index=False, encoding="utf-8-sig"
)
mobility_summary.to_csv(
    OUTPUT_DIR / "student_quartile_mobility.csv", index=False, encoding="utf-8-sig"
)
quartile_descriptive.to_csv(
    OUTPUT_DIR / "quartile_descriptive.csv", index=False, encoding="utf-8-sig"
)

if fit is not None:
    fit.fixed_effects.to_csv(
        OUTPUT_DIR / "fixed_effects.csv", index=False, encoding="utf-8-sig"
    )
    fit.adjusted_quartile_means.to_csv(
        OUTPUT_DIR / "adjusted_quartile_means.csv", index=False, encoding="utf-8-sig"
    )
    fit.module_heterogeneity.to_csv(
        OUTPUT_DIR / "module_heterogeneity.csv", index=False, encoding="utf-8-sig"
    )
    fit.variance_components.to_csv(
        OUTPUT_DIR / "variance_components.csv", index=False, encoding="utf-8-sig"
    )
    fit.diagnostics.to_csv(
        OUTPUT_DIR / "model_diagnostics.csv", index=False, encoding="utf-8-sig"
    )

print(f"Saved summary outputs to: {OUTPUT_DIR}")

Saved summary outputs to: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\artifacts\zpdes_elo_quartile_equity_notebook
